# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all record sets, fields, and columns by their unique `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. This step helps us know which data resources we can extract and analyze.

In [ ]:
# List all available record sets and their field @ids
print("Available Record Sets and their Fields:\n")
record_sets = dataset.metadata.record_sets

for record_set in record_sets:
    print(f"Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    print(f"  Description: {record_set.get('description', 'N/A')}")
    fields = record_set.get('field', [])
    # Fields may be a dict if only one, ensure list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id','')
            field_name = field.get('name','')
        else:
            field_id = field
            field_name = ''
        print(f"    Field @id: {field_id}  Name: {field_name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. 

In [ ]:
# We'll load all record sets into a dictionary of DataFrames, indexed by their @id
record_set_ids = [r['@id'] for r in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    else:
        print(f"  No records found for {record_set_id}.\n")

# For this FAIR^2 dataset, the main experimental protein data table is usually the largest table.
# Let's pick the first non-empty table as the primary record set to explore (change this if you know a specific @id).
main_record_set_id = None
for rid, df in dataframes.items():
    if len(df) > 0:
        main_record_set_id = rid
        break

if main_record_set_id is not None:
    print(f"Primary Record Set for analysis: {main_record_set_id}")
    print("Available columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set found with records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, basic statistics, or grouping. Operations will **reference fields by their `@id` as columns**, as required.

*Example: Filter by peptide count, normalize abundances, and group by gene/protein name if available.*

In [ ]:
# Identify a numeric field and a grouping field by @id (from prior step's columns list)
numeric_field = None
group_field = None

df = dataframes[main_record_set_id].copy()

# Try to automatically select a numeric field with variation (e.g., 'cr:peptideCount', 'cr:abundance', etc.)
for candidate_field in df.columns:
    if df[candidate_field].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[candidate_field]):
        # Avoid null and low-uniqueness fields
        if df[candidate_field].nunique() > 2:
            numeric_field = candidate_field
            break

# Try to select a string field for grouping, e.g., accession or gene name
for candidate_field in df.columns:
    if df[candidate_field].dtype == object and df[candidate_field].nunique() < len(df) // 2:
        group_field = candidate_field
        break

print(f"Selected numeric field for filtering and normalization: {numeric_field}")
print(f"Selected grouping field: {group_field}")

# Set filtering threshold for numeric field
if numeric_field:
    # Do a filter by the mean value
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0

    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}  (Count: {len(filtered_df)})")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by the grouping field and get average normalized value
    if group_field:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean()
            .reset_index()
            .sort_values(numeric_field, ascending=False)
        )
        print(f"\nGrouped data by {group_field}, mean of {numeric_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the extracted and processed data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# Barplot: Mean of numeric_field by group_field (show top 10)
if group_field and numeric_field:
    top_groups = (
        df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    )
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_groups.index, y=top_groups.values)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 mass spectrometry dataset using its Croissant schema and `mlcroissant`, explored its structure using unique `@id` references, and conducted basic exploratory analysis and visualizations. This serves as a reproducible workflow for working with FAIR and Croissant-structured scientific data.